# B Cell L2 Merge + Visualization - PRODUCTION v2.0

**Purpose**: Complete merge pipeline with all P0/P1 fixes + generalized variable naming  
**Version**: 2.0 PRODUCTION (All fixes + marker stability + reusable)  
**Date**: 2026-02-05  
**Author**: r2end

---

## Key Improvements in v2.0

**P0 Critical Fixes:**
1. ✅ QC uses `astype('string')` not `astype(str)`
2. ✅ Category order preserves reference biology (not alphabetical)
3. ✅ Marker expression stability via dedicated obsm matrix

**P1 Strong Recommendations:**
4. ✅ Palette supports >102 categories
5. ✅ Query-only view instead of copy
6. ✅ Legend removes unused categories

**Generalization:**
7. ✅ Variables abstracted (works for L1/L2/L3, any cell type)

---

## Section 1: Imports and Setup

In [104]:
# ===== Imports =====
import sys
import os
from pathlib import Path
import warnings
import json
import gc
from datetime import datetime

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
from scipy.sparse import issparse, csr_matrix

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import scanpy as sc

warnings.filterwarnings('ignore')

print("=" * 80)
print("B Cell Merge + Visualization - PRODUCTION v2.0")
print("=" * 80)
print(f"scanpy: {sc.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"Python: {sys.version}")

B Cell Merge + Visualization - PRODUCTION v2.0
scanpy: 1.11.5
pandas: 2.3.3
numpy: 2.2.5
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]


## Section 2: Configuration (Generalized Variable Naming)

In [105]:
# ===== PATH CONFIGURATION =====
PATH_REF = "/home/h2048/data/py/0203/bcell_scarches_v4_1/models/scanvi_bcell_L2_v2_5_3/reference_with_L2_umap.h5ad"
PATH_QRY = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad"
PATH_OUT = "/home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0"

# ===== OBSKEY CONFIGURATION (Abstracted for reusability) =====
# Reference/Query label keys
OBSKEY_REF_LABEL = "Cell_Type_L2"         # Reference: trained cell type labels
OBSKEY_QRY_PRED = "Cell_Type_L2_pred"     # Query: all predictions (before filtering)
OBSKEY_QRY_FINAL = "Cell_Type_L2_final"   # Query: filtered predictions (confidence >= threshold)

# Metadata keys
OBSKEY_CONF = "mapping_confidence"        # Query: mapping confidence score
OBSKEY_SOURCE = "data_source"             # Merge: reference vs query indicator
OBSKEY_BATCH = "sample"                   # Batch identifier
OBSKEY_TISSUE = "tissue"                  # Tissue source

# ===== VISUALIZATION COLUMN PREFIX =====
# Use prefix to avoid polluting original obs
VIZ_PREFIX = "viz"
VIZ_REF_ONLY = f"{VIZ_PREFIX}__ref_label_only"        # Reference labels visible only on ref cells
VIZ_QRY_FINAL = f"{VIZ_PREFIX}__qry_label_final_only" # Query final labels visible only on qry cells
VIZ_QRY_PRED = f"{VIZ_PREFIX}__qry_label_pred_only"   # Query all predictions visible only on qry cells
VIZ_QRY_CONF = f"{VIZ_PREFIX}__qry_conf_only"         # Query confidence visible only on qry cells

# ===== MARKER GENES =====
MARKER_GENES = [
    "CD19",    # Pan B cell
    "MS4A1",   # CD20, B cell marker
    "CD27",    # Memory B marker
    "IGHD",    # Naive B marker (IgD)
    "IGHM",    # IgM
    "MZB1",    # Plasma cell marker
    "SDC1",    # CD138, Plasma cell
    "JCHAIN",  # Plasma cell
]
MARKER_OBSM_KEY = "marker_expr"  # P0-3: Dedicated obsm for marker expression

# ===== VISUALIZATION SETTINGS =====
DPI = 300
FIGURE_FORMAT = "pdf"

# P1 Fix: PDF rasterization for large scatter plots
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.settings.set_figure_params(dpi=DPI, facecolor='white', format=FIGURE_FORMAT, vector_friendly=False)

# Color schemes
PALETTE_SOURCE = {"reference": "#1f77b4", "query": "#ff7f0e"}
CMAP_CONFIDENCE = "viridis"
CMAP_EXPRESSION = "Reds"

# ===== REPRODUCIBILITY =====
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"\nConfiguration:")
print(f"  Reference: {PATH_REF}")
print(f"  Query: {PATH_QRY}")
print(f"  Output: {PATH_OUT}")
print(f"\n  Label keys:")
print(f"    Reference: {OBSKEY_REF_LABEL}")
print(f"    Query pred: {OBSKEY_QRY_PRED}")
print(f"    Query final: {OBSKEY_QRY_FINAL}")
print(f"\n  Visualization columns use prefix: '{VIZ_PREFIX}__'")


Configuration:
  Reference: /home/h2048/data/py/0203/bcell_scarches_v4_1/models/scanvi_bcell_L2_v2_5_3/reference_with_L2_umap.h5ad
  Query: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad
  Output: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0

  Label keys:
    Reference: Cell_Type_L2
    Query pred: Cell_Type_L2_pred
    Query final: Cell_Type_L2_final

  Visualization columns use prefix: 'viz__'


In [106]:
# ===== Create Output Structure =====
output_dir = Path(PATH_OUT)
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir = output_dir / "figures"
figures_dir.mkdir(exist_ok=True)

print(f"Output directories created:")
print(f"  Main: {output_dir}")
print(f"  Figures: {figures_dir}")

Output directories created:
  Main: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0
  Figures: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/figures


## Section 3: Helper Functions (All Fixes Applied)

In [107]:
# ===== P0-2 FIX: Category order preserves reference biology =====

def build_global_categories(adata, ref_label_key, qry_label_key, source_key="data_source"):
    """
    Build unified category list preserving reference order.
    
    P0-2 FIX: Reference categories come first (in appearance order),
    then query-only categories (sorted). This preserves biological
    hierarchy and ensures stable color mapping.
    
    Args:
        adata: Merged AnnData
        ref_label_key: Reference label column name
        qry_label_key: Query label column name
        source_key: Data source indicator column
    
    Returns:
        list: Ordered category names
    """
    m_ref = adata.obs[source_key].eq("reference")
    m_qry = adata.obs[source_key].eq("query")
    
    # P0 Fix #2 (from v1.1): Use astype('string') not astype(str)
    ref = adata.obs.loc[m_ref, ref_label_key].astype("string").dropna()
    qry = adata.obs.loc[m_qry, qry_label_key].astype("string").dropna()
    
    # Preserve reference appearance order (more natural/biological)
    ref_order = pd.unique(ref)
    
    # Query-only categories (sorted for consistency)
    qry_extra = [x for x in pd.unique(qry) if x not in set(ref_order)]
    
    return list(ref_order) + sorted(qry_extra)


print("✅ build_global_categories: P0-2 fixed (preserves reference order)")

✅ build_global_categories: P0-2 fixed (preserves reference order)


In [108]:
# ===== P1-1 FIX: Palette supports >102 categories =====

def make_palette(categories):
    """
    Create fixed color palette with support for large category counts.
    
    P1-1 FIX: Handles >102 categories with deterministic HSV colors.
    
    Args:
        categories: List of category names
    
    Returns:
        dict: Mapping from category to color
    """
    n = len(categories)
    
    if n <= 20:
        # Use matplotlib tab20 (most distinguishable)
        colors = list(plt.get_cmap("tab20").colors)[:n]
    elif n <= 102:
        # Use scanpy's default 102 colors
        colors = sc.pl.palettes.default_102[:n]
    else:
        # P1-1 FIX: Generate deterministic colors for >102 categories
        colors = [plt.cm.hsv(i / n) for i in range(n)]
        print(f"⚠️  Warning: {n} categories (>102), using HSV colormap")
    
    return dict(zip(categories, colors))


print("✅ make_palette: P1-1 fixed (>102 categories supported)")

✅ make_palette: P1-1 fixed (>102 categories supported)


In [109]:
# ===== P0-3 FIX: Marker expression stability via dedicated obsm =====

def attach_marker_layer(adata, genes, layer_out="marker_expr", prefer_layer="log1p"):
    """
    Create dedicated obsm matrix for marker genes.
    
    P0-3 FIX: Avoids reliance on .raw or .layers which may be lost
    during sc.concat(). Stores expression in .obsm (always preserved).
    
    Args:
        adata: AnnData object
        genes: List of marker gene names
        layer_out: Output obsm key name
        prefer_layer: Preferred source layer
    
    Returns:
        tuple: (adata, available_genes)
    """
    # Filter to available genes
    genes_avail = [g for g in genes if g in adata.var_names]
    
    if not genes_avail:
        print(f"  ⚠️  No marker genes found in var_names")
        return adata, []
    
    # Determine source
    if prefer_layer in (adata.layers or {}):
        X = adata[:, genes_avail].layers[prefer_layer]
        source = f"layer '{prefer_layer}'"
    elif adata.raw is not None and all(g in adata.raw.var_names for g in genes_avail):
        # Fallback to .raw if available
        X = adata.raw[:, genes_avail].X
        source = ".raw.X"
    else:
        X = adata[:, genes_avail].X
        source = ".X"
    
    # Convert to dense array and store in obsm (survives concat)
    adata.obsm[layer_out] = X.toarray() if issparse(X) else np.asarray(X)
    adata.uns[f"{layer_out}_genes"] = genes_avail
    
    print(f"  ✅ Stored {len(genes_avail)} markers in .obsm['{layer_out}'] (from {source})")
    
    return adata, genes_avail


print("✅ attach_marker_layer: P0-3 fixed (marker stability via obsm)")

✅ attach_marker_layer: P0-3 fixed (marker stability via obsm)


In [110]:
# ===== P1-3 FIX: Remove unused categories for clean legends =====

def remove_unused_categories(adata, col):
    """
    Remove unused categories from categorical column.
    
    P1-3 FIX: Prevents legend from showing all GLOBAL_CATS when
    only subset is present in data.
    
    Args:
        adata: AnnData object (modified in-place)
        col: Column name
    """
    if col in adata.obs.columns:
        s = adata.obs[col]
        if isinstance(s.dtype, CategoricalDtype):
            adata.obs[col] = s.cat.remove_unused_categories()


print("✅ remove_unused_categories: P1-3 fixed (clean legends)")

✅ remove_unused_categories: P1-3 fixed (clean legends)


In [111]:
# ===== P1 Fix #5: Rasterization helper =====

def save_rasterized_figure(fig, path, dpi=300, rasterize_scatter=True):
    """
    Save figure with scatter plots rasterized to reduce file size.
    
    Args:
        fig: matplotlib figure
        path: output path
        dpi: resolution
        rasterize_scatter: whether to rasterize scatter collections
    """
    if rasterize_scatter:
        for ax in fig.axes:
            for coll in ax.collections:
                coll.set_rasterized(True)
    
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)


print("✅ save_rasterized_figure: P1-5 (PDF optimization)")

✅ save_rasterized_figure: P1-5 (PDF optimization)


In [112]:
# ===== Summary =====
print("\n" + "=" * 80)
print("HELPER FUNCTIONS LOADED (v2.0 - All Fixes)")
print("=" * 80)
print("P0 Fixes:")
print("  ✅ P0-2: Category order preserves reference biology")
print("  ✅ P0-3: Marker stability via dedicated obsm matrix")
print("\nP1 Fixes:")
print("  ✅ P1-1: Palette supports >102 categories")
print("  ✅ P1-3: Legend removes unused categories")
print("  ✅ P1-5: PDF rasterization for file size")
print("=" * 80)


HELPER FUNCTIONS LOADED (v2.0 - All Fixes)
P0 Fixes:
  ✅ P0-2: Category order preserves reference biology
  ✅ P0-3: Marker stability via dedicated obsm matrix

P1 Fixes:
  ✅ P1-1: Palette supports >102 categories
  ✅ P1-3: Legend removes unused categories
  ✅ P1-5: PDF rasterization for file size


## Section 4: Load Reference Data

In [113]:
print("=" * 80)
print("STEP 1: Load Reference Data")
print("=" * 80)

print(f"\nLoading reference: {PATH_REF}")
adata_ref = sc.read_h5ad(PATH_REF)
adata_ref.var_names_make_unique()
adata_ref.obs_names_make_unique()

print(f"Reference shape: {adata_ref.shape}")
print(f"\nReference .obs columns (first 10):")
print(f"  {list(adata_ref.obs.columns[:10])}")
print(f"\nReference .obsm keys:")
print(f"  {list(adata_ref.obsm.keys())}")

# Check required keys
if OBSKEY_REF_LABEL not in adata_ref.obs.columns:
    raise ValueError(f"Reference missing '{OBSKEY_REF_LABEL}' column!")
if "X_umap" not in adata_ref.obsm:
    raise ValueError("Reference missing 'X_umap' in .obsm!")

print(f"\n✅ Required keys present: '{OBSKEY_REF_LABEL}', 'X_umap'")

print(f"\nReference label distribution (top 10):")
for ct, count in adata_ref.obs[OBSKEY_REF_LABEL].value_counts().head(10).items():
    pct = count / adata_ref.n_obs * 100
    print(f"  {ct}: {count:,} ({pct:.1f}%)")

STEP 1: Load Reference Data

Loading reference: /home/h2048/data/py/0203/bcell_scarches_v4_1/models/scanvi_bcell_L2_v2_5_3/reference_with_L2_umap.h5ad
Reference shape: (11570, 4020)

Reference .obs columns (first 10):
  ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType']

Reference .obsm keys:
  ['X_cnmf_usages', 'X_harmony', 'X_pca', 'X_scANVI_L2', 'X_scanvi', 'X_scanvi_corrected', 'X_scvi', 'X_umap', 'X_umap_scanvi', 'X_umap_scanvi_corrected', 'X_umap_scvi', '_scvi_extra_categorical_covs', '_scvi_extra_continuous_covs']

✅ Required keys present: 'Cell_Type_L2', 'X_umap'

Reference label distribution (top 10):
  GC_B: 3,547 (30.7%)
  Naive_B: 3,012 (26.0%)
  Memory_B: 2,509 (21.7%)
  Plasma: 2,048 (17.7%)
  Atypical_Memory_B: 447 (3.9%)
  Unknown: 7 (0.1%)


In [114]:
# 临时诊断cell（可以插在Section 4之后）
print("Reference actual columns:")
print(adata_ref.obs.columns.tolist())
print("\nQuery actual columns:")
print(adata_query.obs.columns.tolist())

Reference actual columns:
['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType', 'percent.mt', 'percent.ribo', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'decontX_clusters', 'nCount_decontXcounts', 'nFeature_decontXcounts', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'Source', 'Location', 'CellType', 'BroadCellType', 'organism_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomica

## Section 5: P0-3 Fix - Attach Marker Expression to Reference

In [115]:
print("\n" + "=" * 80)
print("P0-3 FIX: Attach Marker Expression (Reference)")
print("=" * 80)

print(f"\nAttaching {len(MARKER_GENES)} marker genes to reference...")
adata_ref, markers_ref = attach_marker_layer(
    adata_ref,
    MARKER_GENES,
    layer_out=MARKER_OBSM_KEY,
    prefer_layer="log1p"  # Adjust if your data uses different layer
)

print(f"\n✅ Reference: {len(markers_ref)} markers stored in .obsm['{MARKER_OBSM_KEY}']")


P0-3 FIX: Attach Marker Expression (Reference)

Attaching 8 marker genes to reference...
  ✅ Stored 8 markers in .obsm['marker_expr'] (from layer 'log1p')

✅ Reference: 8 markers stored in .obsm['marker_expr']


## Section 6: Load Query Data

In [116]:
print("\n" + "=" * 80)
print("STEP 2: Load Query Data")
print("=" * 80)

print(f"\nLoading query: {PATH_QRY}")
adata_query = sc.read_h5ad(PATH_QRY)
adata_query.var_names_make_unique()
adata_query.obs_names_make_unique()

print(f"Query shape: {adata_query.shape}")
print(f"\nQuery .obs columns (first 10):")
print(f"  {list(adata_query.obs.columns[:10])}")
print(f"\nQuery .obsm keys:")
print(f"  {list(adata_query.obsm.keys())}")

# Check required keys
required_query_keys = [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF]
missing_keys = [k for k in required_query_keys if k not in adata_query.obs.columns]
if missing_keys:
    raise ValueError(f"Query missing columns: {missing_keys}")
if "X_umap" not in adata_query.obsm:
    raise ValueError("Query missing 'X_umap' in .obsm!")

print(f"\n✅ Required keys present: {required_query_keys}, 'X_umap'")

print(f"\nQuery label distribution (top 10):")
for ct, count in adata_query.obs[OBSKEY_QRY_FINAL].value_counts().head(10).items():
    pct = count / adata_query.n_obs * 100
    print(f"  {ct}: {count:,} ({pct:.1f}%)")

print(f"\nQuery mapping confidence:")
conf_vals = pd.to_numeric(adata_query.obs[OBSKEY_CONF], errors='coerce')
print(f"  Mean: {conf_vals.mean():.3f}")
print(f"  Median: {conf_vals.median():.3f}")
print(f"  High (>0.9): {(conf_vals > 0.9).sum():,} ({(conf_vals > 0.9).mean()*100:.1f}%)")
print(f"  Low (<0.5): {(conf_vals < 0.5).sum():,} ({(conf_vals < 0.5).mean()*100:.1f}%)")


STEP 2: Load Query Data

Loading query: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3/query_mapped_L2.h5ad
Query shape: (41217, 83690)

Query .obs columns (first 10):
  ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'barcode', 'cellLabel', 'GEO', 'sample', 'dataset', 'tissue', 'tissue_level_2']

Query .obsm keys:
  ['X_scANVI_L2', 'X_scANVI_mapped', 'X_umap', 'X_umap_mapped', '_scvi_extra_categorical_covs', '_scvi_extra_continuous_covs']

✅ Required keys present: ['Cell_Type_L2_pred', 'Cell_Type_L2_final', 'mapping_confidence'], 'X_umap'

Query label distribution (top 10):
  Plasma: 19,902 (48.3%)
  Naive_B: 11,272 (27.3%)
  Memory_B: 9,126 (22.1%)
  Atypical_Memory_B: 630 (1.5%)
  GC_B: 155 (0.4%)
  Unknown: 132 (0.3%)

Query mapping confidence:
  Mean: 0.971
  Median: 1.000
  High (>0.9): 37,497 (91.0%)
  Low (<0.5): 132 (0.3%)


## Section 7: P0-3 Fix - Attach Marker Expression to Query

In [117]:
print("\n" + "=" * 80)
print("P0-3 FIX: Attach Marker Expression (Query)")
print("=" * 80)

print(f"\nAttaching {len(MARKER_GENES)} marker genes to query...")
adata_query, markers_qry = attach_marker_layer(
    adata_query,
    MARKER_GENES,
    layer_out=MARKER_OBSM_KEY,
    prefer_layer="log1p"  # Adjust if your data uses different layer
)

print(f"\n✅ Query: {len(markers_qry)} markers stored in .obsm['{MARKER_OBSM_KEY}']")

# Check consistency
if set(markers_ref) != set(markers_qry):
    print(f"\n⚠️  Warning: Marker availability differs between ref and query")
    print(f"  Ref only: {set(markers_ref) - set(markers_qry)}")
    print(f"  Query only: {set(markers_qry) - set(markers_ref)}")

# Use intersection for merged plotting
markers_common = sorted(set(markers_ref) & set(markers_qry))
print(f"\n✅ Common markers for merge: {len(markers_common)}")
print(f"  {markers_common}")


P0-3 FIX: Attach Marker Expression (Query)

Attaching 8 marker genes to query...
  ✅ Stored 8 markers in .obsm['marker_expr'] (from .X)

✅ Query: 8 markers stored in .obsm['marker_expr']

✅ Common markers for merge: 8
  ['CD19', 'CD27', 'IGHD', 'IGHM', 'JCHAIN', 'MS4A1', 'MZB1', 'SDC1']


## Section 8: Gene Alignment and Merge

In [118]:
# ===== Pre-merge column verification =====
print("\n🔍 Pre-merge column check:")
print(f"\nReference columns containing label-like names:")
ref_label_cols = [c for c in adata_ref.obs.columns if 'type' in c.lower() or 'label' in c.lower()]
print(f"  {ref_label_cols}")

print(f"\nQuery columns containing label-like names:")
qry_label_cols = [c for c in adata_query.obs.columns if 'type' in c.lower() or 'label' in c.lower() or 'pred' in c.lower()]
print(f"  {qry_label_cols}")

print(f"\nExpected columns:")
print(f"  Reference needs: '{OBSKEY_REF_LABEL}'")
print(f"  Query needs: '{OBSKEY_QRY_PRED}', '{OBSKEY_QRY_FINAL}', '{OBSKEY_CONF}'")

# Verify before proceeding
if OBSKEY_REF_LABEL not in adata_ref.obs.columns:
    raise ValueError(f"❌ Reference missing '{OBSKEY_REF_LABEL}'! Available: {list(adata_ref.obs.columns)}")
if OBSKEY_QRY_FINAL not in adata_query.obs.columns:
    raise ValueError(f"❌ Query missing '{OBSKEY_QRY_FINAL}'! Available: {list(adata_query.obs.columns)}")

print("\n✅ Column names verified before merge")


🔍 Pre-merge column check:

Reference columns containing label-like names:
  ['cellType', 'cell_type_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'CellType', 'BroadCellType', 'reannotation_type', 'scanvi_label', 'subject_type', 'marker_doublet_type', '_scvi_labels', 'predicted_labels', 'scanvi_labels', 'cell_type_level_2', 'cell_type_level_3', 'cell_type_celltypist_raw', 'celltypist_predicted_labels', 'cell_type_celltypist_filt', 'labels_for_scanvi', 'cell_type_scanvi_raw', 'cell_type_scanvi_filt', 'cell_type_scanvi_filt_original', 'cell_type_scanvi_corrected', 'cell_type_L3', 'cell_type_expert', 'cell_type_scanvi_pred', 'Cell_Type_L2']

Query columns containing label-like names:
  ['cellLabel', 'data_type', 'species__ontology_label', 'disease__ontology_label', 'organ__ontology_label', 'cell_type__ontology_label', 'biosample_type', 'organism_age__unit_label', 'cell_type', 'geographical_region__ontology_label', 'Cell_type_annotation_level3', 'Cell_type_annotation_le

In [119]:
# ===== CRITICAL DEBUG: Check columns before subsetting =====
print("\n🔍 DEBUG: Checking columns BEFORE gene subsetting...")

print(f"\nadata_ref.obs columns (last 20):")
print(f"  {list(adata_ref.obs.columns[-20:])}")
print(f"\n  Looking for '{OBSKEY_REF_LABEL}': {OBSKEY_REF_LABEL in adata_ref.obs.columns}")

print(f"\nadata_query.obs columns (last 20):")
print(f"  {list(adata_query.obs.columns[-20:])}")
print(f"\n  Looking for '{OBSKEY_QRY_PRED}': {OBSKEY_QRY_PRED in adata_query.obs.columns}")
print(f"  Looking for '{OBSKEY_QRY_FINAL}': {OBSKEY_QRY_FINAL in adata_query.obs.columns}")
print(f"  Looking for '{OBSKEY_CONF}': {OBSKEY_CONF in adata_query.obs.columns}")

if OBSKEY_REF_LABEL not in adata_ref.obs.columns:
    raise ValueError(f"❌ Reference lost '{OBSKEY_REF_LABEL}' before subsetting!")
if OBSKEY_QRY_FINAL not in adata_query.obs.columns:
    raise ValueError(f"❌ Query lost '{OBSKEY_QRY_FINAL}' before subsetting!")


🔍 DEBUG: Checking columns BEFORE gene subsetting...

adata_ref.obs columns (last 20):
  ['cell_type_celltypist_raw', 'celltypist_predicted_labels', 'celltypist_majority_voting', 'celltypist_confidence', 'cell_type_celltypist_filt', 'labels_for_scanvi', 'cell_type_scanvi_raw', 'cell_type_scanvi_filt', 'cell_type_scanvi_filt_original', 'cell_type_scanvi_corrected', 'scanvi_confidence_corrected', 'cell_type_L3', 'cell_type_expert', 'cell_type_scanvi_pred', 'Cell_Type_L2', 'pct_counts_mt', 'stress_score', 'S_score', 'G2M_score', 'phase']

  Looking for 'Cell_Type_L2': True

adata_query.obs columns (last 20):
  ['tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cell_type_query_original', 'pct_counts_mt', 'stress_score', 'S_score', 'G2M_score', '_scvi_batch', 'scanvi_label_existing_cleaned', '_scvi_labels', 'cell_type_mapped', 'mapping_confidence', 'mapping_margin', 'cell_type_final', 'Cell_Type_L2_pred', 'Cel

In [121]:
print("\n" + "=" * 80)
print("STEP 3: Merge Reference + Query (HVG-only, Reference Order)")
print("=" * 80)

# Use reference genes as authoritative list
ref_genes = list(adata_ref.var_names)
common_genes = [g for g in ref_genes if g in adata_query.var_names]
overlap_pct = len(common_genes) / len(ref_genes) * 100

print(f"\nGene overlap:")
print(f"  Reference genes: {len(ref_genes):,}")
print(f"  Common genes: {len(common_genes):,}")
print(f"  Overlap: {overlap_pct:.1f}%")

if overlap_pct < 95.0:
    print(f"⚠️  Warning: Low overlap ({overlap_pct:.1f}%). Merged expression may be incomplete.")

# Subset both to common genes (in reference order)
print(f"\nSubsetting to common genes...")
adata_ref_m = adata_ref[:, common_genes].copy()
adata_qry_m = adata_query[:, common_genes].copy()

print(f"  Reference subset: {adata_ref_m.shape}")
print(f"  Query subset: {adata_qry_m.shape}")

# Keep only essential obsm keys
for adata in [adata_ref_m, adata_qry_m]:
    for key in list(adata.obsm.keys()):
        if key not in ["X_umap", "X_scANVI_L2", MARKER_OBSM_KEY]:
            del adata.obsm[key]
    adata.uns = {}
    if issparse(adata.X):
        adata.X = csr_matrix(adata.X)

print(f"\n✅ Kept obsm keys: X_umap, X_scANVI_L2, {MARKER_OBSM_KEY}")

# Add unique prefixes to obs_names
adata_ref_m.obs_names = [f"ref::{x}" for x in adata_ref_m.obs_names]
adata_qry_m.obs_names = [f"qry::{x}" for x in adata_qry_m.obs_names]

# ===== CRITICAL FIX: Manually add missing obs columns before concat =====
print("\nPreparing obs columns for merge...")

# Reference needs query-specific columns (will be NA for ref cells)
for col in [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF]:
    if col not in adata_ref_m.obs.columns:
        if col == OBSKEY_CONF:
            adata_ref_m.obs[col] = np.nan  # float column
            print(f"  Added '{col}' to reference (NaN)")
        else:
            adata_ref_m.obs[col] = pd.NA  # string column
            print(f"  Added '{col}' to reference (NA)")

# Query needs reference-specific columns (will be NA for query cells)
for col in [OBSKEY_REF_LABEL]:
    if col not in adata_qry_m.obs.columns:
        adata_qry_m.obs[col] = pd.NA  # string column
        print(f"  Added '{col}' to query (NA)")

print(f"\nMerging datasets...")
adata_merged = sc.concat(
    {"reference": adata_ref_m, "query": adata_qry_m},
    axis=0,
    join="outer",  # For var (genes)
    merge="unique",
    label=OBSKEY_SOURCE
)

print(f"✅ Merged shape: {adata_merged.shape}")
print(f"  Using join='outer' to preserve all obs columns from both datasets")

print(f"\nData source distribution:")
print(adata_merged.obs[OBSKEY_SOURCE].value_counts())

# Verify critical obsm integrity
print(f"\nVerifying obsm integrity...")
for key in ["X_umap", MARKER_OBSM_KEY]:
    if key not in adata_merged.obsm:
        raise RuntimeError(f"{key} lost during merge!")
    if adata_merged.obsm[key].shape[0] != adata_merged.n_obs:
        raise RuntimeError(f"{key} shape mismatch after merge!")
    print(f"  ✅ {key}: {adata_merged.obsm[key].shape}")

# Verify marker genes list
if f"{MARKER_OBSM_KEY}_genes" in adata_merged.uns:
    print(f"\n✅ Marker genes list preserved: {len(adata_merged.uns[f'{MARKER_OBSM_KEY}_genes'])} genes")
else:
    print(f"\n⚠️  Warning: Marker genes list not in .uns, using common markers")
    adata_merged.uns[f"{MARKER_OBSM_KEY}_genes"] = markers_common

# Clean up

print(f"\n✅ Merge complete!")


STEP 3: Merge Reference + Query (HVG-only, Reference Order)

Gene overlap:
  Reference genes: 4,020
  Common genes: 3,955
  Overlap: 98.4%

Subsetting to common genes...
  Reference subset: (11570, 3955)
  Query subset: (41217, 3955)

✅ Kept obsm keys: X_umap, X_scANVI_L2, marker_expr

Preparing obs columns for merge...
  Added 'Cell_Type_L2_pred' to reference (NA)
  Added 'Cell_Type_L2_final' to reference (NA)
  Added 'mapping_confidence' to reference (NaN)
  Added 'Cell_Type_L2' to query (NA)

Merging datasets...
✅ Merged shape: (52787, 3955)
  Using join='outer' to preserve all obs columns from both datasets

Data source distribution:
data_source
query        41217
reference    11570
Name: count, dtype: int64

Verifying obsm integrity...
  ✅ X_umap: (52787, 2)
  ✅ marker_expr: (52787, 8)

⚠️  Warning: Marker genes list not in .uns, using common markers

✅ Merge complete!


In [123]:
# ===== CRITICAL: Verify required columns exist after merge =====
print("\nVerifying required columns after merge...")

required_ref_cols = [OBSKEY_REF_LABEL]
required_qry_cols = [OBSKEY_QRY_PRED, OBSKEY_QRY_FINAL, OBSKEY_CONF]

missing_cols = []
for col in required_ref_cols + required_qry_cols:
    if col not in adata_merged.obs.columns:
        missing_cols.append(col)

if missing_cols:
    print(f"\n❌ ERROR: Missing columns after merge: {missing_cols}")
    print(f"\nAvailable columns in adata_merged.obs:")
    print(f"  {list(adata_merged.obs.columns)}")
    
    print(f"\n💡 Hint: Check that OBSKEY_* variables in Section 2 match your actual h5ad column names.")
    print(f"Expected columns:")
    print(f"  OBSKEY_REF_LABEL = '{OBSKEY_REF_LABEL}'")
    print(f"  OBSKEY_QRY_PRED = '{OBSKEY_QRY_PRED}'")
    print(f"  OBSKEY_QRY_FINAL = '{OBSKEY_QRY_FINAL}'")
    print(f"  OBSKEY_CONF = '{OBSKEY_CONF}'")
    
    raise ValueError(
        f"Required columns {missing_cols} not found after merge. "
        f"Please reload your h5ad files and check actual column names."
    )
else:
    # Define masks for verification
    ref_mask = adata_merged.obs[OBSKEY_SOURCE] == "reference"
    qry_mask = adata_merged.obs[OBSKEY_SOURCE] == "query"
    
    print("✅ All required columns present:")
    for col in required_ref_cols:
        n_valid = adata_merged.obs.loc[ref_mask, col].notna().sum()
        print(f"  ✓ {col} (reference): {n_valid} cells")
    for col in required_qry_cols:
        n_valid = adata_merged.obs.loc[qry_mask, col].notna().sum()
        print(f"  ✓ {col} (query): {n_valid} cells")


Verifying required columns after merge...
✅ All required columns present:
  ✓ Cell_Type_L2 (reference): 11570 cells
  ✓ Cell_Type_L2_pred (query): 41217 cells
  ✓ Cell_Type_L2_final (query): 41217 cells
  ✓ mapping_confidence (query): 41217 cells


## Section 9: Build Global Categories + Fixed Palette

In [124]:
print("\n" + "=" * 80)
print("STEP 4: P0-2 FIX - Build Global Categories + Fixed Palette")
print("=" * 80)

print("\nBuilding global cell type categories...")
GLOBAL_CATS = build_global_categories(
    adata_merged,
    OBSKEY_REF_LABEL,
    OBSKEY_QRY_FINAL,
    OBSKEY_SOURCE
)

print(f"\nTotal unique cell types: {len(GLOBAL_CATS)}")
print(f"Categories (first 20): {GLOBAL_CATS[:20]}")
if len(GLOBAL_CATS) > 20:
    print(f"  ... and {len(GLOBAL_CATS) - 20} more")

# Create fixed palette
print(f"\nCreating fixed color palette...")
CT_PALETTE = make_palette(GLOBAL_CATS)
print(f"✅ Palette created: {len(CT_PALETTE)} colors")
if len(GLOBAL_CATS) <= 20:
    print(f"  Type: tab20")
elif len(GLOBAL_CATS) <= 102:
    print(f"  Type: default_102")
else:
    print(f"  Type: HSV (deterministic)")


STEP 4: P0-2 FIX - Build Global Categories + Fixed Palette

Building global cell type categories...

Total unique cell types: 6
Categories (first 20): ['Atypical_Memory_B', 'GC_B', 'Memory_B', 'Plasma', 'Naive_B', 'Unknown']

Creating fixed color palette...
✅ Palette created: 6 colors
  Type: tab20


## Section 10: Create Cleaned Visualization Columns

In [125]:
print("\n" + "=" * 80)
print("STEP 5: Create Cleaned Visualization Columns")
print("=" * 80)

# P1 Fix #4: Use pandas Series masks (not numpy arrays)
ref_mask = adata_merged.obs[OBSKEY_SOURCE].eq("reference")
qry_mask = adata_merged.obs[OBSKEY_SOURCE].eq("query")

print(f"\nMasks:")
print(f"  Reference cells: {ref_mask.sum():,}")
print(f"  Query cells: {qry_mask.sum():,}")

# 1. Query-only L2 Final
print(f"\nCreating {VIZ_QRY_FINAL}...")
adata_merged.obs[VIZ_QRY_FINAL] = pd.Series(
    pd.NA,
    index=adata_merged.obs_names,
    dtype="string"  # P0 Fix #2
)

if OBSKEY_QRY_FINAL in adata_merged.obs.columns:
    qry_ser = adata_merged.obs[OBSKEY_QRY_FINAL].astype("string")  # P0 Fix #2
    adata_merged.obs.loc[qry_mask, VIZ_QRY_FINAL] = qry_ser[qry_mask]

# P0 Fix #1: Force to use GLOBAL_CATS
adata_merged.obs[VIZ_QRY_FINAL] = pd.Categorical(
    adata_merged.obs[VIZ_QRY_FINAL],
    categories=GLOBAL_CATS
)

print(f"  ✅ Non-NA values: {adata_merged.obs[VIZ_QRY_FINAL].notna().sum()}")

# 2. Query-only confidence
print(f"\nCreating {VIZ_QRY_CONF}...")
conf_qry_only = np.full(adata_merged.n_obs, np.nan, dtype=float)

if OBSKEY_CONF in adata_merged.obs.columns:
    conf_vals = pd.to_numeric(
        adata_merged.obs.loc[qry_mask, OBSKEY_CONF],
        errors='coerce'
    ).values
    conf_qry_only[qry_mask] = conf_vals

adata_merged.obs[VIZ_QRY_CONF] = conf_qry_only
print(f"  ✅ Finite values: {np.isfinite(conf_qry_only).sum()}")

# 3. Reference-only L2
print(f"\nCreating {VIZ_REF_ONLY}...")
adata_merged.obs[VIZ_REF_ONLY] = pd.Series(
    pd.NA,
    index=adata_merged.obs_names,
    dtype="string"  # P0 Fix #2
)

if OBSKEY_REF_LABEL in adata_merged.obs.columns:
    ref_ser = adata_merged.obs[OBSKEY_REF_LABEL].astype("string")  # P0 Fix #2
    adata_merged.obs.loc[ref_mask, VIZ_REF_ONLY] = ref_ser[ref_mask]

# P0 Fix #1: Force to use GLOBAL_CATS
adata_merged.obs[VIZ_REF_ONLY] = pd.Categorical(
    adata_merged.obs[VIZ_REF_ONLY],
    categories=GLOBAL_CATS
)

print(f"  ✅ Non-NA values: {adata_merged.obs[VIZ_REF_ONLY].notna().sum()}")

# 4. Query-only predictions (all, not filtered)
print(f"\nCreating {VIZ_QRY_PRED}...")
adata_merged.obs[VIZ_QRY_PRED] = pd.Series(
    pd.NA,
    index=adata_merged.obs_names,
    dtype="string"
)

if OBSKEY_QRY_PRED in adata_merged.obs.columns:
    pred_ser = adata_merged.obs[OBSKEY_QRY_PRED].astype("string")
    adata_merged.obs.loc[qry_mask, VIZ_QRY_PRED] = pred_ser[qry_mask]

adata_merged.obs[VIZ_QRY_PRED] = pd.Categorical(
    adata_merged.obs[VIZ_QRY_PRED],
    categories=GLOBAL_CATS
)

print(f"  ✅ Non-NA values: {adata_merged.obs[VIZ_QRY_PRED].notna().sum()}")

print(f"\n✅ All visualization columns created!")


STEP 5: Create Cleaned Visualization Columns

Masks:
  Reference cells: 11,570
  Query cells: 41,217

Creating viz__qry_label_final_only...
  ✅ Non-NA values: 41217

Creating viz__qry_conf_only...
  ✅ Finite values: 41217

Creating viz__ref_label_only...
  ✅ Non-NA values: 11570

Creating viz__qry_label_pred_only...
  ✅ Non-NA values: 41217

✅ All visualization columns created!


## Section 11: Save Merged Data and Configuration

In [126]:
print("\n" + "=" * 80)
print("STEP 6: Save Merged Data")
print("=" * 80)

merged_output = output_dir / "reference_plus_query_merged_v2_0.h5ad"
print(f"\nSaving merged data: {merged_output}")
adata_merged.write_h5ad(merged_output, compression='gzip')
file_size = merged_output.stat().st_size / 1024**3
print(f"✅ Saved ({file_size:.2f} GB)")


STEP 6: Save Merged Data

Saving merged data: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/reference_plus_query_merged_v2_0.h5ad
✅ Saved (0.06 GB)


In [127]:
print("\n" + "=" * 80)
print("STEP 7: Save Configuration")
print("=" * 80)

viz_config = {
    "pipeline": "B_Cell_Merge_Visualization",
    "version": "2.0_PRODUCTION_ALL_FIXES",
    "timestamp": datetime.now().isoformat(),
    "input_reference": str(PATH_REF),
    "input_query": str(PATH_QRY),
    "output_dir": str(PATH_OUT),
    "obskeys": {
        "ref_label": OBSKEY_REF_LABEL,
        "qry_pred": OBSKEY_QRY_PRED,
        "qry_final": OBSKEY_QRY_FINAL,
        "confidence": OBSKEY_CONF,
        "source": OBSKEY_SOURCE,
        "batch": OBSKEY_BATCH
    },
    "viz_columns": {
        "prefix": VIZ_PREFIX,
        "ref_only": VIZ_REF_ONLY,
        "qry_final": VIZ_QRY_FINAL,
        "qry_pred": VIZ_QRY_PRED,
        "qry_conf": VIZ_QRY_CONF
    },
    "merged_shape": {
        "n_obs": int(adata_merged.n_obs),
        "n_vars": int(adata_merged.n_vars)
    },
    "data_source_counts": adata_merged.obs[OBSKEY_SOURCE].value_counts().to_dict(),
    "global_categories": GLOBAL_CATS,
    "n_categories": len(GLOBAL_CATS),
    "palette_type": "tab20" if len(GLOBAL_CATS) <= 20 else ("default_102" if len(GLOBAL_CATS) <= 102 else "hsv"),
    "marker_genes": adata_merged.uns.get(f"{MARKER_OBSM_KEY}_genes", []),
    "marker_obsm_key": MARKER_OBSM_KEY,
    "fixes_applied": {
        "P0_1": "Global categories + fixed color palette",
        "P0_2_v1": "astype('string') for proper NA handling",
        "P0_2_v2": "Category order preserves reference biology",
        "P0_3": "Marker stability via dedicated obsm matrix",
        "P1_1": "Palette supports >102 categories",
        "P1_3": "Legend removes unused categories",
        "P1_4": "Pandas Series masks",
        "P1_5": "PDF rasterization enabled"
    }
}

config_path = output_dir / "merge_visualization_config_v2_0.json"
with open(config_path, 'w') as f:
    json.dump(viz_config, f, indent=2)
print(f"✅ Configuration saved: {config_path}")


STEP 7: Save Configuration
✅ Configuration saved: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/merge_visualization_config_v2_0.json


## Section 12: Visualization 1 - Merged Overview (6-panel)

In [128]:
print("\n" + "=" * 80)
print("STEP 8: Generate Visualizations")
print("=" * 80)

print("\n8.1: Merged Overview (6-panel)")
print("-" * 80)

fig, axes = plt.subplots(2, 3, figsize=(24, 16))

# Panel 1: Data source
sc.pl.umap(
    adata_merged,
    color=OBSKEY_SOURCE,
    ax=axes[0, 0],
    show=False,
    title="Data Source",
    palette=PALETTE_SOURCE,
    frameon=False,
    s=20
)

# Panel 2: Reference labels - FIXED palette
sc.pl.umap(
    adata_merged,
    color=VIZ_REF_ONLY,
    ax=axes[0, 1],
    show=False,
    title="Reference Labels (Reference Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 3: Query predictions - FIXED palette
sc.pl.umap(
    adata_merged,
    color=VIZ_QRY_FINAL,
    ax=axes[0, 2],
    show=False,
    title="Query Predictions (Query Only)",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 4: Batch
if OBSKEY_BATCH in adata_merged.obs.columns:
    sc.pl.umap(
        adata_merged,
        color=OBSKEY_BATCH,
        ax=axes[1, 0],
        show=False,
        title="Batch",
        frameon=False,
        s=20,
        legend_loc=None  # P1 Fix #6: too many batches
    )

# Panel 5: Confidence (query only)
sc.pl.umap(
    adata_merged,
    color=VIZ_QRY_CONF,
    ax=axes[1, 1],
    show=False,
    title="Mapping Confidence (Query Only)",
    cmap=CMAP_CONFIDENCE,
    vmin=0,
    vmax=1,
    frameon=False,
    s=20,
    na_color='lightgray'
)

# Panel 6: Marker expression (P0-3 FIX: from obsm)
marker_genes_list = adata_merged.uns.get(f"{MARKER_OBSM_KEY}_genes", [])
if "CD19" in marker_genes_list:
    marker_idx = marker_genes_list.index("CD19")
    expr = adata_merged.obsm[MARKER_OBSM_KEY][:, marker_idx]
    
    # Manual scatter plot
    umap_coords = adata_merged.obsm["X_umap"]
    scatter = axes[1, 2].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c=expr,
        cmap=CMAP_EXPRESSION,
        s=20,
        rasterized=True
    )
    axes[1, 2].set_title("CD19 Expression", fontsize=14)
    axes[1, 2].set_xlabel("UMAP1")
    axes[1, 2].set_ylabel("UMAP2")
    axes[1, 2].axis('off')
    plt.colorbar(scatter, ax=axes[1, 2], fraction=0.046, pad=0.04)
else:
    axes[1, 2].text(
        0.5, 0.5,
        "CD19\nNot Available",
        ha='center',
        va='center',
        transform=axes[1, 2].transAxes,
        fontsize=14
    )
    axes[1, 2].axis('off')

plt.tight_layout()
output_path = figures_dir / f"merged_overview_6panel_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"✅ Saved: {output_path.name}")


STEP 8: Generate Visualizations

8.1: Merged Overview (6-panel)
--------------------------------------------------------------------------------
✅ Saved: merged_overview_6panel_v2_0.pdf


## Section 13: Visualization 2 - Side-by-Side Comparison

In [129]:
print("\n8.2: Side-by-Side Comparison (Reference vs Query)")
print("-" * 80)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: Reference labels - FIXED palette
sc.pl.umap(
    adata_merged,
    color=VIZ_REF_ONLY,
    ax=axes[0],
    show=False,
    title="Reference: Original Labels",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=25,
    na_color='lightgray'
)

# Right: Query predictions - FIXED palette (same colors!)
sc.pl.umap(
    adata_merged,
    color=VIZ_QRY_FINAL,
    ax=axes[1],
    show=False,
    title="Query: Predicted Labels",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=25,
    na_color='lightgray'
)

plt.tight_layout()
output_path = figures_dir / f"reference_vs_query_sidebyside_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"✅ Saved: {output_path.name}")


8.2: Side-by-Side Comparison (Reference vs Query)
--------------------------------------------------------------------------------
✅ Saved: reference_vs_query_sidebyside_v2_0.pdf


## Section 14: Visualization 3 - Marker Gene Panel

In [130]:
print("\n8.3: B Cell Marker Gene Panel (P0-3 FIX: from obsm)")
print("-" * 80)

marker_genes_list = adata_merged.uns.get(f"{MARKER_OBSM_KEY}_genes", [])

if len(marker_genes_list) > 0:
    n_markers = len(marker_genes_list)
    ncols = 4
    nrows = int(np.ceil(n_markers / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes
    
    umap_coords = adata_merged.obsm["X_umap"]
    
    for i, gene in enumerate(marker_genes_list):
        expr = adata_merged.obsm[MARKER_OBSM_KEY][:, i]
        
        # Manual scatter plot (stable, no reliance on .raw/.layers)
        scatter = axes[i].scatter(
            umap_coords[:, 0],
            umap_coords[:, 1],
            c=expr,
            cmap=CMAP_EXPRESSION,
            s=30,
            rasterized=True
        )
        axes[i].set_title(f"{gene} Expression", fontsize=12, fontweight='bold')
        axes[i].set_xlabel("UMAP1", fontsize=10)
        axes[i].set_ylabel("UMAP2", fontsize=10)
        axes[i].axis('off')
        plt.colorbar(scatter, ax=axes[i], fraction=0.046, pad=0.04)
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    output_path = figures_dir / f"bcell_markers_panel_v2_0.{FIGURE_FORMAT}"
    save_rasterized_figure(fig, output_path, dpi=DPI)
    print(f"✅ Saved: {output_path.name}")
else:
    print("⚠️  No marker genes available in .obsm")


8.3: B Cell Marker Gene Panel (P0-3 FIX: from obsm)
--------------------------------------------------------------------------------
✅ Saved: bcell_markers_panel_v2_0.pdf


## Section 15: Visualization 4 - Query-Only Detailed View

In [131]:
print("\n8.4: Query-Only Detailed View")
print("-" * 80)

# P1-2 FIX: Use view instead of copy
query_cells = adata_merged.obs[OBSKEY_SOURCE] == "query"
adata_query_subset = adata_merged[query_cells]  # View, not copy

# P1-3 FIX: Remove unused categories for clean legends
remove_unused_categories(adata_query_subset, VIZ_QRY_PRED)
remove_unused_categories(adata_query_subset, VIZ_QRY_FINAL)

fig, axes = plt.subplots(2, 2, figsize=(18, 18))

# Panel 1: All predictions
sc.pl.umap(
    adata_query_subset,
    color=VIZ_QRY_PRED,
    ax=axes[0, 0],
    show=False,
    title="All Predictions",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=30
)

# Panel 2: Filtered predictions
sc.pl.umap(
    adata_query_subset,
    color=VIZ_QRY_FINAL,
    ax=axes[0, 1],
    show=False,
    title="Filtered Predictions (Confidence >= 0.5)",
    legend_loc="right margin",
    palette=CT_PALETTE,  # P0 Fix #1
    frameon=False,
    s=30
)

# Panel 3: Confidence heatmap
sc.pl.umap(
    adata_query_subset,
    color=VIZ_QRY_CONF,
    ax=axes[1, 0],
    show=False,
    title="Mapping Confidence Score",
    cmap=CMAP_CONFIDENCE,
    vmin=0,
    vmax=1,
    frameon=False,
    s=30
)

# Panel 4: Confidence histogram
conf_vals = adata_query_subset.obs[VIZ_QRY_CONF].values
conf_vals = conf_vals[np.isfinite(conf_vals)]

axes[1, 1].hist(
    conf_vals,
    bins=50,
    edgecolor='black',
    alpha=0.7,
    color='steelblue'
)
axes[1, 1].axvline(
    0.5,
    color='red',
    linestyle='--',
    linewidth=2,
    label='Threshold = 0.5'
)
axes[1, 1].set_xlabel('Mapping Confidence', fontsize=12)
axes[1, 1].set_ylabel('Number of Cells', fontsize=12)
axes[1, 1].set_title('Confidence Distribution', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(alpha=0.3)
axes[1, 1].spines['top'].set_visible(False)
axes[1, 1].spines['right'].set_visible(False)

plt.tight_layout()
output_path = figures_dir / f"query_detailed_view_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"✅ Saved: {output_path.name}")

# Clean up view
gc.collect()


8.4: Query-Only Detailed View
--------------------------------------------------------------------------------
✅ Saved: query_detailed_view_v2_0.pdf


34932

## Section 16: Visualization 5 - High-Resolution Individual Exports

In [132]:
print("\n8.5: High-Resolution Individual Exports")
print("-" * 80)

export_params = {
    'frameon': False,
    's': 50,
    'show': False
}

# 1. Data source
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=OBSKEY_SOURCE,
    ax=ax,
    title="",
    palette=PALETTE_SOURCE,
    **export_params
)
output_path = figures_dir / f"umap_data_source_highres_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"  ✅ Saved: {output_path.name}")

# 2. Reference labels (on data legend)
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=VIZ_REF_ONLY,
    ax=ax,
    title="",
    legend_loc="on data",  # P1 Fix #6
    legend_fontsize=10,
    legend_fontoutline=2,
    palette=CT_PALETTE,  # P0 Fix #1
    na_color='lightgray',
    **export_params
)
output_path = figures_dir / f"umap_ref_labels_highres_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"  ✅ Saved: {output_path.name}")

# 3. Query labels (on data legend)
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=VIZ_QRY_FINAL,
    ax=ax,
    title="",
    legend_loc="on data",  # P1 Fix #6
    legend_fontsize=10,
    legend_fontoutline=2,
    palette=CT_PALETTE,  # P0 Fix #1
    na_color='lightgray',
    **export_params
)
output_path = figures_dir / f"umap_qry_labels_highres_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"  ✅ Saved: {output_path.name}")

# 4. Confidence
fig, ax = plt.subplots(figsize=(10, 8))
sc.pl.umap(
    adata_merged,
    color=VIZ_QRY_CONF,
    ax=ax,
    title="",
    cmap=CMAP_CONFIDENCE,
    vmin=0,
    vmax=1,
    na_color='lightgray',
    **export_params
)
output_path = figures_dir / f"umap_confidence_highres_v2_0.{FIGURE_FORMAT}"
save_rasterized_figure(fig, output_path, dpi=DPI)
print(f"  ✅ Saved: {output_path.name}")

print(f"\n✅ High-resolution exports complete!")


8.5: High-Resolution Individual Exports
--------------------------------------------------------------------------------
  ✅ Saved: umap_data_source_highres_v2_0.pdf
  ✅ Saved: umap_ref_labels_highres_v2_0.pdf
  ✅ Saved: umap_qry_labels_highres_v2_0.pdf
  ✅ Saved: umap_confidence_highres_v2_0.pdf

✅ High-resolution exports complete!


## Section 17: Quality Control Analysis

In [133]:
print("\n" + "=" * 80)
print("STEP 9: Quality Control Analysis")
print("=" * 80)

# Extract query cells for QC
query_mask_qc = adata_merged.obs[OBSKEY_SOURCE] == "query"

# P0-1 FIX: Use astype('string') not astype(str)
df_qc = pd.DataFrame({
    'cell_type': adata_merged.obs.loc[query_mask_qc, VIZ_QRY_FINAL].astype("string"),  # P0-1 FIX
    'confidence': pd.to_numeric(
        adata_merged.obs.loc[query_mask_qc, VIZ_QRY_CONF],
        errors='coerce'
    )
}).dropna()

print(f"\nQC dataset: {len(df_qc):,} cells with valid data")

# Summary statistics
summary_stats = df_qc.groupby('cell_type')['confidence'].agg([
    ('Count', 'count'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(3)

# Add percentage
summary_stats['Percentage'] = (
    summary_stats['Count'] / summary_stats['Count'].sum() * 100
).round(2)

# Sort by count
summary_stats = summary_stats.sort_values('Count', ascending=False)

print("\n" + "=" * 80)
print("CONFIDENCE SUMMARY BY CELL TYPE")
print("=" * 80)
print(summary_stats.to_string())
print("=" * 80)

# Save to CSV
csv_path = output_dir / "confidence_summary_by_celltype_v2_0.csv"
summary_stats.to_csv(csv_path)
print(f"\n✅ Summary table saved: {csv_path}")


STEP 9: Quality Control Analysis

QC dataset: 41,217 cells with valid data

CONFIDENCE SUMMARY BY CELL TYPE
                   Count   Mean  Median    Std    Min  Max  Percentage
cell_type                                                             
Plasma             19902  0.998   1.000  0.024  0.501  1.0       48.29
Naive_B            11272  0.957   1.000  0.100  0.500  1.0       27.35
Memory_B            9126  0.944   0.998  0.113  0.500  1.0       22.14
Atypical_Memory_B    630  0.860   0.932  0.155  0.503  1.0        1.53
GC_B                 155  0.914   0.997  0.145  0.511  1.0        0.38
Unknown              132  0.453   0.463  0.042  0.313  0.5        0.32

✅ Summary table saved: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/confidence_summary_by_celltype_v2_0.csv


In [134]:
# Plot: Top 10 cell types confidence distribution
print("\nGenerating confidence plots (top 10 cell types)...")

top_cts = summary_stats.head(10).index
df_plot_top = df_qc[df_qc['cell_type'].isin(top_cts)].copy()

# Sort by median confidence
ct_order = df_plot_top.groupby('cell_type')['confidence'].median().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Violin plot
sns.violinplot(
    data=df_plot_top,
    x='cell_type',
    y='confidence',
    order=ct_order,
    ax=axes[0],
    palette='Set2'
)
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
axes[0].set_title('Confidence by Cell Type (Top 10, Violin)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cell Type', fontsize=12)
axes[0].set_ylabel('Mapping Confidence', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot
sns.boxplot(
    data=df_plot_top,
    x='cell_type',
    y='confidence',
    order=ct_order,
    ax=axes[1],
    palette='Set2'
)
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold = 0.5')
axes[1].set_title('Confidence by Cell Type (Top 10, Box)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cell Type', fontsize=12)
axes[1].set_ylabel('Mapping Confidence', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
output_path = figures_dir / f"confidence_by_celltype_top10_v2_0.{FIGURE_FORMAT}"
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f"✅ Saved: {output_path.name}")


Generating confidence plots (top 10 cell types)...
✅ Saved: confidence_by_celltype_top10_v2_0.pdf


## Section 18: Final Summary Report

In [135]:
print("\n" + "=" * 80)
print("FINAL SUMMARY - v2.0 PRODUCTION")
print("=" * 80)

ref_mask_summary = adata_merged.obs[OBSKEY_SOURCE] == "reference"
qry_mask_summary = adata_merged.obs[OBSKEY_SOURCE] == "query"

print(f"\nMerged Dataset:")
print(f"  Total cells: {adata_merged.n_obs:,}")
print(f"  Genes (HVG): {adata_merged.n_vars:,}")
print(f"  Reference cells: {ref_mask_summary.sum():,}")
print(f"  Query cells: {qry_mask_summary.sum():,}")

print(f"\nColor Palette:")
print(f"  Global categories: {len(GLOBAL_CATS)}")
print(f"  Palette type: {viz_config['palette_type']}")
print(f"  ✅ Consistent colors across all plots")
print(f"  ✅ Reference biology preserved in category order")

print(f"\nMarker Genes:")
print(f"  Available: {len(marker_genes_list)}")
print(f"  Storage: .obsm['{MARKER_OBSM_KEY}']")
print(f"  ✅ Stable (survives concat)")

print(f"\nQuery Mapping Summary (top 10):")
for ct, count in adata_merged.obs.loc[qry_mask_summary, VIZ_QRY_FINAL].value_counts().head(10).items():
    pct = count / qry_mask_summary.sum() * 100
    print(f"  {ct}: {count:,} ({pct:.1f}%)")

print(f"\nQuery Confidence:")
conf_qry = pd.to_numeric(
    adata_merged.obs.loc[qry_mask_summary, VIZ_QRY_CONF],
    errors='coerce'
)
print(f"  Mean: {conf_qry.mean():.3f}")
print(f"  Median: {conf_qry.median():.3f}")
print(f"  High (>0.9): {(conf_qry > 0.9).sum():,} ({(conf_qry > 0.9).mean()*100:.1f}%)")
print(f"  Low (<0.5): {(conf_qry < 0.5).sum():,} ({(conf_qry < 0.5).mean()*100:.1f}%)")

print(f"\nOutput Files:")
print(f"  Merged data: {merged_output}")
print(f"  Config: {config_path}")
print(f"  QC summary: {csv_path}")
print(f"  Figures: {figures_dir}/ ({len(list(figures_dir.glob('*')))} files)")

print(f"\n✨ Fixes Applied (v2.0):")
print(f"  [OK] P0-1: QC uses astype('string') not astype(str)")
print(f"  [OK] P0-2: Category order preserves reference biology")
print(f"  [OK] P0-3: Marker stability via dedicated obsm matrix")
print(f"  [OK] P1-1: Palette supports >102 categories")
print(f"  [OK] P1-2: Query-only view instead of copy")
print(f"  [OK] P1-3: Legend removes unused categories")
print(f"  [OK] Generalized: Variables abstracted (L1/L2/L3 compatible)")

print("\n" + "=" * 80)
print("SUCCESS - Ready for Publication!")
print("=" * 80)


FINAL SUMMARY - v2.0 PRODUCTION

Merged Dataset:
  Total cells: 52,787
  Genes (HVG): 3,955
  Reference cells: 11,570
  Query cells: 41,217

Color Palette:
  Global categories: 6
  Palette type: tab20
  ✅ Consistent colors across all plots
  ✅ Reference biology preserved in category order

Marker Genes:
  Available: 8
  Storage: .obsm['marker_expr']
  ✅ Stable (survives concat)

Query Mapping Summary (top 10):
  Plasma: 19,902 (48.3%)
  Naive_B: 11,272 (27.3%)
  Memory_B: 9,126 (22.1%)
  Atypical_Memory_B: 630 (1.5%)
  GC_B: 155 (0.4%)
  Unknown: 132 (0.3%)

Query Confidence:
  Mean: 0.971
  Median: 1.000
  High (>0.9): 37,497 (91.0%)
  Low (<0.5): 132 (0.3%)

Output Files:
  Merged data: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/reference_plus_query_merged_v2_0.h5ad
  Config: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/merge_visualization_config_v2_0.json
  QC summary: /home/h2048/data/py/0204/scarches_mapping_L2_v2_5_3_MERGE_v2_0/confidence_su

---

## Usage Notes

### For L1/L3 or Other Cell Types

To adapt this notebook for different label levels or cell types:

1. **Update Section 2 (Configuration)**:
   ```python
   # For L3 analysis:
   OBSKEY_REF_LABEL = "Cell_Type_L3"
   OBSKEY_QRY_PRED = "Cell_Type_L3_pred"
   OBSKEY_QRY_FINAL = "Cell_Type_L3_final"
   
   # For T/NK cells:
   PATH_REF = ".../tnk_reference.h5ad"
   PATH_QRY = ".../tnk_query.h5ad"
   MARKER_GENES = ["CD3D", "CD4", "CD8A", "GNLY", ...]  # T/NK markers
   ```

2. **Run all cells** - No other changes needed!

### Key Improvements Over v1.1

1. **P0-1**: QC analysis no longer creates "nan" string category
2. **P0-2**: Category order respects biological hierarchy from reference
3. **P0-3**: Marker expression completely independent of .raw/.layers concat behavior
4. **P1-1**: Handles >102 cell types (important for L3/atlas)
5. **P1-2**: Memory efficient (views vs copies)
6. **P1-3**: Clean legends (no unused categories)
7. **Generalized**: Variable naming allows easy adaptation to other analyses

---